<a target="_blank" href="https://colab.research.google.com/github/MScEcologyAndDataScienceUCL/BIOS0032_AI4Environment/blob/main/TrackingData/practical.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Week 8: Machine Learning for tracking data

What we will learn

In this weeks practical we will explore how machine learning algorithms can be applied to data collected from tracking devices (GPS).
We will learn to:

- Visualise tracking data
- Extract and visualise some useful features/metrics from tracking data (speed, tortuosity)
- Fit an unsupervised mixture model (kmeans, GMM) to those data
- Visualise the resulting classification on the tracking data

If time permits we may also explore:

- Training a Hidden Markov Model to classify tracking data
- Using a neural network to classifying tracking data

**Make sure you have selected the R runtime before proceeding**

## Setup

Please run the following cell to install the required R dependencies and download the data needed for the practical.

```r
system(
  "curl -LsSf https://github.com/MScEcologyAndDataScienceUCL/BIOS0032_AI4Environment/raw/refs/heads/main/TrackingData/setup.sh | sh",
  intern = TRUE
)
```

## Load libraries

```r
# Load libraries
library(data.table)
library(ggplot2)
library(dplyr)
library(moveHMM)
library(dplyr)
library(mapdata)
library(geosphere)
library(mclust)
library(factoextra)
```

## Data

Load the tracking data and make a quick plot of all the tracks.

```r
dat <- fread("anon_gps_tracks_with_dive.csv")

print(nrow(dat))
head(dat)
```

```r
# Quick plot of all tracks, separated by species
ggplot(dat, aes(x = lon, y = lat, group = species, color = species)) +
  geom_point() +
  facet_wrap(~species)
```

We've got a variety of columns including species (Guillemots [COGU], Shags [EUSH] and Razorbills [RAZO]), bird (the individual), latitude (lat), longitude (lon), altitude (alt), a unix timestamp (unix), the colony (colony2)

Might be nicer to see these on a map...

```r
ggplot(dat, aes(lon, lat, color = species, group = species)) +
  annotation_map(map_data("world")) + # Add the map as a base layer before the points
  geom_point(pch = 16) +
  coord_quickmap() # Sets aspect ratio
```

Or facetted by species...

```r
ggplot(dat, aes(lon, lat, color = species, group = species)) +
  annotation_map(map_data("world")) + # Add the map as a base layer before the points
  geom_point(pch = 16) +
  coord_quickmap() +
  facet_wrap(~species)
```

How many birds are there for each species at each colony?

```r
dat %>%
  group_by(species, colony2) %>%
  summarise(nbird = n_distinct(bird))
```

## Metrics/Features

## Speed

One useful metric for thinking about tracking data is the speed with which an individual is moving.
When moving slowly we may think it is behaving differently from when is moving quickly (e.g. resting vs commuting).
Let's calculate speed for all individuals.

Calculate speed for every individual...

```r
# Function to calculate speed as dist in m / time
haversine_speed <- function(lat2, lat1, long2, long1, time2, time1) {
  dist_in_m = geosphere::distHaversine(
    cbind(long1, lat1),
    cbind(long2, lat2),
    r = 6378137
  )
  timediff <- time2 - time1
  return(dist_in_m / timediff)
}

# Apply that function to each individual, arranged by time, along the lagged positions
dat_with_speed = dat %>%
  group_by(bird) %>%
  arrange(bird, unix) %>%
  mutate(
    speed_ms = haversine_speed(lat, lag(lat), lon, lag(lon), unix, lag(unix))
  )
```

Have a quick look at the speeds..

```r
head(dat_with_speed$speed_ms)
```

Plot the speed distributions...

```r
ggplot(dat_with_speed, aes(x = speed_ms, group = species, color = species)) +
  geom_histogram(binwidth = 0.1, alpha = 0.6)
```

Might be useful to log transform data

```r
ggplot(dat_with_speed, aes(x = speed_ms, group = species, fill = species)) +
  geom_histogram(binwidth = 0.1, alpha = 0.6) +
  scale_x_log10()
```

Or split by species

```r
ggplot(dat_with_speed, aes(x = speed_ms, group = species, fill = species)) +
  geom_histogram(binwidth = 0.1, alpha = 0.6) +
  scale_x_log10() +
  facet_wrap(~species)
```

We could try plotting the tracks with speed to color the positions (to see where they are moving fast/slow)...

```r
ggplot(dat_with_speed, aes(lon, lat, color = speed_ms, group = species)) +
  annotation_map(map_data("world")) + # Add the map as a base layer before the points
  geom_point(pch = 16, alpha = 0.1) +
  coord_quickmap() +
  facet_wrap(~species) +
  scale_color_continuous(name = "speed", trans = "log")
```

...
but it's very hard to discriminate anything at this scale - let's focus on one bird...

```r
one_bird = subset(dat_with_speed, bird == 1)

ggplot(one_bird, aes(lon, lat, color = speed_ms, group = species)) +
  annotation_map(map_data("world")) + # Add the map as a base layer before the points
  geom_point(pch = 16, alpha = 0.1) +
  coord_quickmap() +
  scale_color_continuous(name = "speed", trans = "log")
```

Here we can see some faster (lighter) locations and some slower (darker) locations, possibly where the bird is resting?

Let's have a look at it's speed distribution

```r
# Might be useful to log transform data
ggplot(one_bird, aes(x = speed_ms, group = species, fill = species)) +
  geom_histogram(binwidth = 0.1, alpha = 0.6) +
  scale_x_log10()
```

We could try clustering on speed, perhaps using kmeans?

```r
kmeans_result = stats::kmeans(
  log10(na.omit(one_bird$speed_ms) + 1),
  2,
  nstart = 25
)

print(summary(kmeans_result))
print(kmeans_result$center)
print(table(kmeans_result$cluster))
```

Need to consider how many clusters there are...

Lets use the factoextra package to work out how many clusters are most likely....

```r
# Dropping NAs (here the first speed)
one_bird$log_speed_ms_s <- scale(log10(one_bird$speed_ms + 1))

library(factoextra)

fviz_nbclust(na.omit(one_bird$log_speed_ms_s), kmeans, method = "wss") +
  geom_vline(xintercept = 3, linetype = 2)
```

This suggests that (for this bird) 3 clusters might be the best fit, let's rerun kmeans with 3 centres..

```r
kmeans_result = stats::kmeans(
  log10(na.omit(one_bird$speed_ms) + 1),
  3,
  nstart = 25
)
```

We can then apply these back to the bird data.
Note the appended NA needed as the NA speed (first speed) was dropped above)

```r
one_bird$behaviour = c(NA, kmeans_result$cluster)
```

And plot these behaviours on the birds track

```r
ggplot(one_bird, aes(lon, lat, color = factor(behaviour), group = species)) +
  annotation_map(map_data("world")) + # Add the map as a base layer before the points
  geom_point(pch = 16) +
  coord_quickmap()
```

## Metrics - tortuosity

Tortuosity or straightness, can be calculated in a number of ways.
Here we're going to use turning angles and step-lengths along our paths.

For this, let's lean on one of the many available R packages for analysing tracks (moveHMM)

https://cran.r-project.org/web/packages/moveHMM/vignettes/moveHMM-guide.pdf

We can use the package moveHMM to calculate step-length and turning-angle

```r
data <- prepData(one_bird, type = "LL", coordNames = c("lon", "lat"))
```

Quick plot of the track with step-length and turning-angle calculated by moveHMM

```r
plot(data, compact = TRUE, ask = FALSE)
```

### Trajectory straightness index, E-max

(from https://github.com/JimMcL/trajr/blob/master/R/straightness.R)

Emax, the maximum expected displacement, is a single-valued measure of straightness defined by (Cheung, Zhang, Stricker, & Srinivasan, 2007).
Emax-a is a dimensionless, scale-independent measure of the maximum possible expected displacement.
Emax-b is \code{Emax-a * mean step length}, and gives the maximum possible expected displacement in spatial units.
Values closer to 0 are *more\* sinuous, while larger values (approaching infinity) are straighter.

Calculate the strightness index (e-max) by calculating the mean cos(turning_angle) and the mean step-length within a window (here 20 samples).

```r
data <- data %>%
  mutate(
    mean_angle = frollmean(
      cos(angle),
      n = 20,
      fill = NA,
      align = "left",
      na.rm = TRUE
    ),
    mean_step = frollmean(
      step,
      n = 20,
      fill = NA,
      align = "left",
      na.rm = TRUE
    ),
    emaxb = mean_step * mean_angle / (1 - mean_angle)
  )
```

```r
plot(log10(data$step), log10(data$emaxb))
```

We could now apply kmean to this metric, but let's apply it to speed and straightness together.

```r
data$log_speed_ms_s <- scale(log10(data$speed_ms + 1))
data$log_emaxb_s <- scale(log10(data$emaxb + 1))

data = filter(data, !is.na(log_emaxb_s))

fviz_nbclust(
  na.omit(cbind(data$log_speed_ms_s, data$log_emaxb_s)),
  kmeans,
  method = "silhouette" # (can also try 'wss', 'silhouette', 'gap_stat')
) +
  geom_vline(xintercept = 3, linetype = 2)
```

```r
kmeans_result = stats::kmeans(
  na.omit(cbind(data$speed_ms, data$emaxb)),
  3,
  nstart = 25
)
```

```r
data$behaviour = -1
data$behaviour[complete.cases(data)] = kmeans_result$cluster
```

```r
ggplot(data, aes(x, y, color = factor(behaviour), group = species)) +
  annotation_map(map_data("world")) + # Add the map as a base layer before the points
  geom_point(pch = 16) +
  coord_quickmap()
```

```r
kmeans_result$centers
```

# Other metrics?

Can you think/calculate other metrics to include in the models?

# Gaussian mixture models

The same process can be done with a gaussian mixture model (GMM) using the `mclust` package

First we can fit a range of models to the data and see how they compare

```r
BIC <- mclustBIC(cbind(data$log_speed_ms_s))
plot(BIC)
```

Then use the results of this to select a model...

```r
# mod1 <- Mclust(cbind(data$log_speed_ms_s, data$log_emaxb_s), x = BIC)
mod1 <- Mclust(data$log_speed_ms_s, x = BIC)
summary(mod1, parameters = TRUE)
```

And we can plot the classifications and uncertainty from this model..

```r
plot(mod1, what = "classification")
```

```r
plot(mod1, what = "uncertainty")
```

We can extract the model classifications and apply them to the trajectory...

```r
data$gmm_behaviour = mod1$classification
```

And plot the resulting classifications...

```r
ggplot(data, aes(x, y, color = factor(behaviour), group = species)) +
  annotation_map(map_data("world")) + # Add the map as a base layer before the points
  geom_point(pch = 16) +
  coord_quickmap()
```

Importantly, the GMM is a probabilistic model, we can obtain the probabilities of each sample coming from each class:

```r
head(mod1$z)
```

And we could then use those to explore locations the model is more confident in

```r
data$max_z = apply(mod1$z, 1, max)

# Filter out classifications with p<0.9
ggplot(
  filter(data, max_z > 0.9),
  aes(x, y, color = factor(gmm_behaviour), group = species, size = max_z)
) +
  annotation_map(map_data("world")) + # Add the map as a base layer before the points
  geom_point(pch = 16) +
  coord_quickmap()
```

## Hidden Markov Models

We can also fit Hidden-markov models which better capture the temporal dymnamics of a time-series.
These models (or similar 'state-space models') are now frequently used to model animal movement data.
Either to capture behavioural-states (as here) or to estimate errors associated with movement states.

These models simultaneously fit distributions for the metrics we care about (step-length, turning-angle) while also assuming that these distirbutions differ with the animals behaviour.
We can specifiy how many of these 'states' we think there should be and fit a model.

The model will estimate both parameters for the mean/sd of each state, but also the transition probabilities between them (the probability of being in state X/Y at the next iteration, when you are in state X or Y now)T

```r
###
# Hidden Markov Models
###

# Priors

# Starting values for the step length parameters
# initial means (one for each state)
stepMean0 <- c(0.1, 1.0, 1.0)

# initial standard deviations (one for each state) stepPar0 <- c(stepMean0, stepSD0)
stepSD0 <- c(0.1, 0.7, 1.0)

### starting values for step angle distribution parameters
stepPar0 <- c(stepMean0, stepSD0)

# zeromass0 <- c(0.1, 0.05) # step zero-mass

# turning angle mean of each state
angleMean0 <- c(pi, 0, pi)

# angle concentration
kappa0 <- c(0.01, 8, 0.5)

# starting values for the parameters of the turning angle distributions
anglePar0 <- c(angleMean0, kappa0)

## call to fitting function
# ***
# ** One-state model, e.g. a NULL model with no transitions (should be roughly equivalent to a GMM)
# ***
m_1 <- fitHMM(
  data = na.omit(data),
  nbStates = 1,
  stepPar0 = stepPar0[c(1, 4)],
  anglePar0 = anglePar0[c(1, 4)],
  formula = ~1,
)

### get info from model
m_1
```

Fit a two-state model using two parameters from the stepLength and angle priors

```r
# Note the plotting doesn't work (as 1-state model i think!)
#plot(m_1, plotCI=TRUE, ask = F)

###
# 2-state model
###

## call to fitting function
m_2 <- fitHMM(
  data = na.omit(data),
  nbStates = 2,
  stepPar0 = stepPar0[c(1:2, 4:5)],
  anglePar0 = anglePar0[c(1:2, 4:5)],
  formula = ~1
)

### get info from model
m_2
plot(m_2, plotCI = TRUE, ask = F)
```

Fit a three-state model using all of the parameters

```r
###
# 3-state model
###

## call to fitting function
m_3 <- fitHMM(
  data = na.omit(data),
  nbStates = 3,
  stepPar0 = stepPar0,
  anglePar0 = anglePar0,
  formula = ~1
)

### get info from model
m_3
plot(m_3, plotCI = TRUE, ask = F)
```

And compare the AIC of each model (lower is better)

```r
# Compare AICs - here a 2-state model is favoured
AIC(m_1, m_2, m_3)
```

Look at the summary output for the HMM.
What do you think the 'transition probabilities' are?
How could we use these probabilities?

## Some further things to try

### Other birds

- Can you apply a classification to all of the individuals from one colony/species?
- How do the number of states differ between individuals (does this change if run the models across all of them individually/together?)

### HMMs on haggis

There's a walk through of how to apply an HMM to 'Haggis' tracks here:
https://cran.r-project.org/web/packages/moveHMM/vignettes/moveHMM-example.pdf

### Acceleration data

- You could try applying these to the acceleration data we collected previously, that data is here:
  https://www.dropbox.com/s/8p1v0xzvuy1q0qf/all_csv_data_msc_datascience_2023.csv.zip?dl=0 (45Mb compressed, uncompresses to 700Mb)

### Deep learning

- In Browning et al (2018), we were able to predict dives using just the lat/lon data - can you predict dives using these data?
- To constuct matrices for predictions, we used an embedding matrix (a matrix where each row is from a different time point across a rolling window).
  In `r`, you can construct these using `embed`
- Can you predict `species` using the movement data?

# How well can we do with a Random Forest?

```r
install.packages("ranger") # Can be slow on colab
```

```r
library(dplyr)
library(ranger)
```

```r
# assume your data frame is called df
rf_dat <- dat %>%
  mutate(
    is_dive_1m = factor(is_dive_1m, levels = c(FALSE, TRUE)),
    bird = factor(bird),
    species = factor(species),
    year = factor(year),
    colony2 = factor(colony2),
    hour = as.integer(format(date_time, "%H")),
    yday = as.integer(format(date_time, "%j"))
  ) %>%
  select(
    is_dive_1m,
    lat,
    lon,
    alt,
    coverage_ratio,
    bird,
    species,
    year,
    colony2,
    hour,
    yday
  ) %>%
  na.omit()

set.seed(1)

# simple random row split
train_id <- sample(seq_len(nrow(rf_dat)), size = 0.8 * nrow(rf_dat))
train_dat <- rf_dat[train_id, ]
test_dat <- rf_dat[-train_id, ]

rf_mod <- ranger(
  is_dive_1m ~ .,
  data = train_dat,
  probability = TRUE,
  num.trees = 500,
  importance = "impurity"
)

# predicted probabilities for TRUE
pred_prob <- predict(rf_mod, data = test_dat)$predictions[, "TRUE"]

# convert to class using 0.5 threshold
pred_class <- ifelse(pred_prob > 0.5, "TRUE", "FALSE") |>
  factor(levels = c("FALSE", "TRUE"))
```

```r
# confusion matrix
table(observed = test_dat$is_dive_1m, predicted = pred_class)
```

### Variable Importance

```r
plot_vi = data.frame(
  variable = names(rf_mod$variable.importance),
  var.imp = as.numeric(rf_mod$variable.importance)
)

head(plot_vi)

ggplot(plot_vi, aes(x = variable, y = var.imp)) +
  geom_bar(stat = "identity") +
  coord_flip()

as.numeric(rf_mod$variable.importance)
```

```r

install.packages("pROC") # Can be slow on colab
```

What does the ROC/AUC look like?

```r
library(pROC)

roc_obj <- roc(test_dat$is_dive_1m, pred_prob) auc(roc_obj) plot(roc_obj)

Clearly the model is good, but the optimal threshold is not 0.5

opt <- coords( roc_obj, x = "best", best.method = "youden", transpose = FALSE )

opt
```

Extract this threshold and classify using it

```r
best_thresh <- opt$threshold

pred_class <- ifelse(pred_prob >= best_thresh, TRUE, FALSE)
pred_class <- factor(pred_class, levels = c(FALSE, TRUE))
```

Much better apparent performance...

```r
confusion <- table(observed = test_dat$is_dive_1m, predicted = pred_class)

confusion
```

Add predictions to test data frame

```r
library(dplyr)

test_pred <- test_dat %>%
  mutate(pred_prob = pred_prob, pred_class = pred_prob >= best_thresh)
```

What do these predictions look like per bird?

```r
conf_by_bird <- test_pred %>%
  group_by(species, bird) %>%
  summarise(
    TP = sum(pred_class == TRUE & is_dive_1m == TRUE),
    TN = sum(pred_class == FALSE & is_dive_1m == FALSE),
    FP = sum(pred_class == TRUE & is_dive_1m == FALSE),
    FN = sum(pred_class == FALSE & is_dive_1m == TRUE),
    sensitivity = TP / (TP + FN),
    specificity = TN / (TN + FP),
    precision = TP / (TP + FP),
    accuracy = (TP + TN) / (TP + TN + FP + FN),
    .groups = "drop"
  )

conf_by_bird
```